In [ ]:
# Bioreactor Data Viewer — Jupyter Notebook with ipywidgets
import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from collections import OrderedDict

# Force inline rendering in Jupyter notebooks
pio.renderers.default = 'notebook'

# ---- Load & prepare data ----
working_dir = Path.cwd()
DATA_FILE = Path("Project 1 - DTB/bioreaction_remaining.csv")
SEARCH_PATHS = (
    root / directory / DATA_FILE
    for directory in ("input-data", "init_data")
    for root in (working_dir, *working_dir.parents)
)
DATA_PATH = next((path for path in SEARCH_PATHS if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(f"Could not find {DATA_FILE} under input-data or init_data.")
df = pd.read_csv(DATA_PATH, dtype={"batch_id": str})
df["dead"] = df["tcd"] - df["vcd"]
df["viability"] = 100 * df["vcd"] / df["tcd"]

test_ids = set(df.loc[df["to_predict"], "batch_id"])

MEAS = ["vcd", "tcd", "substrate", "byproduct"]
FEEDS = ["feed_substrate", "feed_b", "feed_c", "feed_d"]

def num(x):
    return None if pd.isna(x) else round(float(x), 6)

batches = {}
summary = []
for bid, g in sorted(df.groupby("batch_id"), key=lambda kv: int(kv[0].replace("Batch", ""))):
    g = g.sort_values("elapsed_time")
    split = "test" if bid in test_ids else "train"
    series = {}
    for col in MEAS + ["dead", "viability", "volume"] + FEEDS:
        s = g[["elapsed_time", col]].dropna()
        series[col] = [s["elapsed_time"].round(5).tolist(), s[col].round(6).tolist()]
    batches[bid] = {
        "split": split,
        "series": series,
        "raw": g.drop(columns=["dead", "viability"]).replace({np.nan: None}).to_dict(orient="records"),
    }

    m = g.dropna(subset=["vcd"])
    first = m.iloc[0] if len(m) else None
    last = m.iloc[-1] if len(m) else None
    row = {
        "batch": bid,
        "split": split,
        "n_meas": int(len(m)),
        "t_end": num(g["elapsed_time"].max()),
        "vcd0": num(first["vcd"]) if first is not None else None,
        "substrate0": num(first["substrate"]) if first is not None else None,
        "byproduct0": num(first["byproduct"]) if first is not None else None,
        "vcd_max": num(m["vcd"].max()) if split == "train" else None,
        "t_vcd_max": num(m.loc[m["vcd"].idxmax(), "elapsed_time"]) if split == "train" else None,
        "vcd_end": num(last["vcd"]) if split == "train" else None,
        "viab_end": num(last["viability"]) if split == "train" else None,
        "byproduct_max": num(m["byproduct"].max()) if split == "train" else None,
        "substrate_min": num(m["substrate"].min()) if split == "train" else None,
        "vol0": num(g["volume"].dropna().iloc[0]),
        "vol_end": num(g["volume"].dropna().iloc[-1]),
    }
    for f in FEEDS:
        row[f"{f}_total"] = num(g[f].sum())
        row[f"{f}_n"] = int(g[f].notna().sum())
    summary.append(row)

IDS = list(batches.keys())
SUM = {r["batch"]: r for r in summary}

VARS = [
    {"k": "vcd", "label": "VCD", "unit": "cells", "on": True},
    {"k": "tcd", "label": "TCD", "unit": "cells", "on": True},
    {"k": "dead", "label": "Dead (TCD−VCD)", "unit": "cells"},
    {"k": "viability", "label": "Viability %", "unit": "%"},
    {"k": "substrate", "label": "Substrate", "unit": "conc", "on": True},
    {"k": "byproduct", "label": "Byproduct", "unit": "conc", "on": True},
    {"k": "volume", "label": "Volume", "unit": "vol"},
    {"k": "feed_substrate", "label": "Feed substrate", "feed": True},
    {"k": "feed_b", "label": "Feed B", "feed": True},
    {"k": "feed_c", "label": "Feed C", "feed": True},
    {"k": "feed_d", "label": "Feed D", "feed": True},
]

SUMCOLS = [
    ["batch", "Batch"], ["split", "Split"], ["n_meas", "# meas"], ["t_end", "t end"],
    ["vcd0", "VCD₀"], ["substrate0", "Subst₀"], ["byproduct0", "Byprod₀"],
    ["vcd_max", "VCD max"], ["t_vcd_max", "t @ VCD max"], ["vcd_end", "VCD end"],
    ["viab_end", "Viab end %"], ["byproduct_max", "Byprod max"],
    ["substrate_min", "Subst min"], ["vol0", "Vol₀"], ["vol_end", "Vol end"],
    ["feed_substrate_total", "ΣFeed S"], ["feed_substrate_n", "#S"],
    ["feed_b_total", "ΣFeed B"], ["feed_b_n", "#B"],
    ["feed_c_total", "ΣFeed C"], ["feed_c_n", "#C"],
    ["feed_d_total", "ΣFeed D"], ["feed_d_n", "#D"],
]
print(f"Loaded {len(batches)} batches ({len(batches)-len(test_ids)} train, {len(test_ids)} test)")

In [ ]:
# ---- Color & formatting helpers ----
VIRIDIS = [[68, 1, 84], [59, 82, 139], [33, 145, 140], [94, 201, 98], [253, 231, 37]]

def batch_color(id_):
    """Unique hue per batch (golden angle spacing)."""
    i = IDS.index(id_)
    return f"hsl({(i * 137.508) % 360:.0f}, 65%, {40 if i % 2 else 50}%)"

def viridis(t):
    """Viridis colour scale interpolator."""
    t = max(0, min(1, t))
    s, i = t * 4, min(3, int(t * 4))
    f = s - i
    c = [round(VIRIDIS[i][j] + f * (VIRIDIS[i + 1][j] - VIRIDIS[i][j])) for j in range(3)]
    return f"rgb({c[0]},{c[1]},{c[2]})"

def fmt(v):
    if v is None or not np.isfinite(v):
        return "–"
    if abs(v) >= 100:
        return f"{v:.1f}"
    if abs(v) >= 1:
        return f"{v:.2f}"
    return f"{v:.3g}"

print("Helpers ready.")

In [ ]:
# ---- Global selection state ----
train_ids = [i for i in IDS if batches[i]["split"] == "train"]
selected = set(train_ids[:6])

In [ ]:
# =============================================================
#  SIDEBAR — batch list with checkboxes, search, quick-select
# =============================================================

search_box = widgets.Text(
    placeholder="Filter, e.g. 12 or Batch3",
    layout=widgets.Layout(width="100%")
)
batch_checkboxes = widgets.VBox()
stats_label = widgets.HTML()

def update_stats():
    n_train = sum(1 for i in IDS if batches[i]["split"] == "train")
    n_test = len(IDS) - n_train
    stats_label.value = (
        f"<span style='color:#6b7280'>"
        f"{len(IDS)} batches ({n_train} train, {n_test} test) · "
        f"{len(selected)} selected</span>"
    )

def build_checkboxes():
    q = search_box.value.strip().lower()
    rows = []
    for id_ in IDS:
        if q and q not in id_.lower():
            continue
        s = batches[id_]["split"]
        cb = widgets.Checkbox(
            value=id_ in selected,
            description=f"{id_} ({s})",
            indent=False,
            layout=widgets.Layout(margin="0 0 2px 0"),
        )
        cb.observe(lambda change, bid=id_: on_checkbox(change, bid), "value")
        rows.append(cb)
    batch_checkboxes.children = tuple(rows) if rows else (widgets.HTML("<i style='color:#9ca3af'>No batches match</i>"),)
    update_stats()

def on_checkbox(change, bid):
    if change["new"]:
        selected.add(bid)
    else:
        selected.discard(bid)
    update_stats()
    refresh_active_tab()

search_box.observe(lambda _: build_checkboxes(), "value")

# Quick-select buttons
btn_all = widgets.Button(description="All", layout=widgets.Layout(width="auto"))
btn_none = widgets.Button(description="None", layout=widgets.Layout(width="auto"))
btn_train = widgets.Button(description="Train", layout=widgets.Layout(width="auto"))
btn_test = widgets.Button(description="Test", layout=widgets.Layout(width="auto"))
btn_rand5 = widgets.Button(description="Random 5", layout=widgets.Layout(width="auto"))

def quick_select(kind):
    global selected
    if kind == "all":
        selected = set(IDS)
    elif kind == "none":
        selected = set()
    elif kind == "train" or kind == "test":
        selected = {i for i in IDS if batches[i]["split"] == kind}
    elif kind == "rand":
        import random
        selected = set(random.sample(IDS, min(5, len(IDS))))
    build_checkboxes()
    refresh_active_tab()

btn_all.on_click(lambda _: quick_select("all"))
btn_none.on_click(lambda _: quick_select("none"))
btn_train.on_click(lambda _: quick_select("train"))
btn_test.on_click(lambda _: quick_select("test"))
btn_rand5.on_click(lambda _: quick_select("rand"))

sidebar = widgets.VBox([
    widgets.HTML("<h3 style='margin:4px 0;text-transform:uppercase;font-size:11px;letter-spacing:.05em;color:#6b7280'>Batches</h3>"),
    widgets.HBox([btn_all, btn_none, btn_train, btn_test, btn_rand5]),
    search_box,
    batch_checkboxes,
    stats_label,
], layout=widgets.Layout(width="260px", padding="8px"))

update_stats()
build_checkboxes()
display(sidebar)  # We'll build the full layout later

In [ ]:
# =============================================================
#  PLOTS TAB
# =============================================================

# Variable chips
var_checkboxes = []
var_chips_box = widgets.HBox()

def build_var_chips():
    global var_checkboxes
    var_checkboxes = []
    for v in VARS:
        cb = widgets.Checkbox(value=v.get("on", False), description=v["label"], indent=False)
        cb.layout.margin = "0 8px 0 0"
        cb.observe(lambda change, vv=v: refresh_active_tab(), "value")
        var_checkboxes.append((v, cb))
    var_chips_box.children = tuple(cb for _, cb in var_checkboxes)

def active_vars():
    return [v for v, cb in var_checkboxes if cb.value]

build_var_chips()

# Other plot controls
mode_dd = widgets.Dropdown(
    options=[("Overlay: one panel per variable", "overlay"),
             ("Separate: one row per batch", "separate")],
    value="overlay"
)
feed_mode_dd = widgets.Dropdown(
    options=[("Individual additions", "events"), ("Cumulative", "cum")],
    value="events"
)
color_by_dd = widgets.Dropdown(
    options=[("Batch (distinct colors)", "batch"), ("Train / test", "split")] +
            [(f"Metric: {lbl}", k) for k, lbl in SUMCOLS[2:]],
    value="batch"
)
markers_cb = widgets.Checkbox(value=True, description="markers", indent=False)
logy_cb = widgets.Checkbox(value=False, description="log y (cells)", indent=False)
show_legend_cb = widgets.Checkbox(value=True, description="legend", indent=False)

for w in [mode_dd, feed_mode_dd, color_by_dd, markers_cb, logy_cb, show_legend_cb]:
    w.observe(lambda _: refresh_active_tab(), "value")

grad_legend = widgets.HTML("", layout=widgets.Layout(visibility="hidden"))

def color_fn():
    by = color_by_dd.value
    sel_ids = [i for i in IDS if i in selected]
    if by == "batch":
        grad_legend.layout.visibility = "hidden"
        return lambda id_: batch_color(id_)
    if by == "split":
        grad_legend.layout.visibility = "hidden"
        return lambda id_: "#d97706" if batches[id_]["split"] == "test" else "#2563eb"
    vals = [SUM[i][by] for i in sel_ids if SUM[i].get(by) is not None]
    if not vals:
        grad_legend.layout.visibility = "hidden"
        return lambda id_: "#bbb"
    lo, hi = min(vals), max(vals)
    grad_legend.value = (
        f"<div style='display:flex;align-items:center;gap:6px;font-size:12px;color:#6b7280'>"
        f"<span>{fmt(lo)}</span>"
        f"<div style='height:10px;width:160px;border-radius:3px;"
        f"background:linear-gradient(90deg,#440154,#3b528b,#21918c,#5ec962,#fde725)'></div>"
        f"<span>{fmt(hi)}</span></div>"
    )
    grad_legend.layout.visibility = "visible"
    def mapper(id_):
        val = SUM[id_].get(by)
        return "#bbb" if val is None else viridis((val - lo) / (hi - lo) if hi > lo else 0.5)
    return mapper

def series_for(id_, v):
    x, y = batches[id_]["series"][v["k"]]
    if v.get("feed") and feed_mode_dd.value == "cum":
        s = 0
        y_cum = []
        for a in y:
            s += a
            y_cum.append(s)
        return [0] + x, [0] + y_cum
    return x, y

def make_trace(id_, v, axis_idx, color, showleg):
    x, y = series_for(id_, v)
    cum = v.get("feed") and feed_mode_dd.value == "cum"
    mk = markers_cb.value
    test = batches[id_]["split"] == "test"
    return go.Scatter(
        x=x, y=y, name=id_, legendgroup=id_, showlegend=showleg,
        xaxis="x" + (str(axis_idx) if axis_idx else ""),
        yaxis="y" + (str(axis_idx) if axis_idx else ""),
        mode="markers" if (v.get("feed") and not cum) else ("lines+markers" if mk else "lines"),
        line=dict(color=color, width=1.6, dash="dot" if test else "solid", shape="hv" if cum else "linear"),
        marker=dict(color=color, size=6 if (v.get("feed") and not cum) else 4),
        hovertemplate=f"<b>%{{text}}</b><br>t=%{{x:.2f}} d<br>{v['label']}=%{{y:.4g}}<extra></extra>",
        text=[f"{id_} ({batches[id_]['split']})"] * len(x),
    )

plots_output = widgets.Output()

def render_plots():
    plots_output.clear_output(wait=True)
    with plots_output:
        vars_ = active_vars()
        sel_ids = [i for i in IDS if i in selected]
        if not vars_ or not sel_ids:
            display(HTML("<p style='color:#6b7280'>Select at least one batch and one variable.</p>"))
            return
        color = color_fn()
        cols = min(len(vars_), 3)

        if mode_dd.value == "overlay":
            rows_ = (len(vars_) + cols - 1) // cols
            fig = make_subplots(
                rows=rows_, cols=cols,
                subplot_titles=[v["label"] for v in vars_],
                vertical_spacing=0.18 if rows_ > 1 else 0.1,
                horizontal_spacing=0.08,
            )
            for vi, v in enumerate(vars_):
                r, c = vi // cols + 1, vi % cols + 1
                for id_ in sel_ids:
                    fig.add_trace(make_trace(id_, v, str(vi + 1) if vi else "", color(id_), vi == 0), row=r, col=c)
            logy = logy_cb.value
            for vi, v in enumerate(vars_):
                axis_num = str(vi + 1) if vi else ""
                fig.update_xaxes(title_text="time [d]", gridcolor="#eee", zeroline=False, row=vi // cols + 1, col=vi % cols + 1)
                fig.update_yaxes(
                    gridcolor="#eee", zeroline=False,
                    type="log" if (logy and v["unit"] == "cells") else "linear",
                    row=vi // cols + 1, col=vi % cols + 1
                )
            fig.update_layout(
                height=330 * rows_, margin=dict(l=50, r=10, t=30, b=35),
                showlegend=show_legend_cb.value, legend=dict(font=dict(size=11)),
                hovermode="closest", font=dict(size=11),
                paper_bgcolor="#fff", plot_bgcolor="#fff",
            )
            display(fig)
        else:  # separate mode
            sep_cols = min(len(vars_), 4)
            for id_ in sel_ids:
                s = SUM[id_]
                rows_sep = (len(vars_) + sep_cols - 1) // sep_cols
                fig = make_subplots(
                    rows=rows_sep, cols=sep_cols,
                    subplot_titles=[v["label"] for v in vars_],
                    vertical_spacing=0.18 if rows_sep > 1 else 0.1,
                    horizontal_spacing=0.08,
                )
                for vi, v in enumerate(vars_):
                    r, c = vi // sep_cols + 1, vi % sep_cols + 1
                    fig.add_trace(make_trace(id_, v, str(vi + 1) if vi else "", color(id_), False), row=r, col=c)
                logy = logy_cb.value
                for vi, v in enumerate(vars_):
                    fig.update_xaxes(title_text="time [d]", gridcolor="#eee", zeroline=False, row=vi // sep_cols + 1, col=vi % sep_cols + 1)
                    fig.update_yaxes(
                        gridcolor="#eee", zeroline=False,
                        type="log" if (logy and v["unit"] == "cells") else "linear",
                        row=vi // sep_cols + 1, col=vi % sep_cols + 1
                    )
                fig.update_layout(
                    height=220 * rows_sep, margin=dict(l=50, r=10, t=50, b=35),
                    showlegend=False, hovermode="closest", font=dict(size=11),
                    paper_bgcolor="#fff", plot_bgcolor="#fff",
                    title=dict(
                        text=f"{id_} — {s['split']} · VCD₀ {fmt(s['vcd0'])} · VCD max {fmt(s['vcd_max'])} · ΣFeed B {fmt(s['feed_b_total'])}",
                        font=dict(size=12)
                    ),
                )
                display(fig)

# Controls panel
plots_controls = widgets.VBox([
    widgets.HBox([
        widgets.VBox([widgets.HTML("<span style='font-size:11px;text-transform:uppercase;color:#6b7280'>Variables</span>"), var_chips_box]),
        widgets.VBox([widgets.HTML("<span style='font-size:11px;text-transform:uppercase;color:#6b7280'>Layout</span>"), mode_dd]),
        widgets.VBox([widgets.HTML("<span style='font-size:11px;text-transform:uppercase;color:#6b7280'>Feeds shown as</span>"), feed_mode_dd]),
    ]),
    widgets.HBox([
        widgets.VBox([widgets.HTML("<span style='font-size:11px;text-transform:uppercase;color:#6b7280'>Color by</span>"), color_by_dd, grad_legend]),
        widgets.VBox([widgets.HTML("<span style='font-size:11px;text-transform:uppercase;color:#6b7280'>Options</span>"),
                      markers_cb, logy_cb, show_legend_cb]),
    ]),
], layout=widgets.Layout(padding="8px", border="1px solid #e3e6eb", border_radius="8px"))

plots_tab = widgets.VBox([plots_controls, plots_output])

In [ ]:
# =============================================================
#  SUMMARY TABLE TAB
# =============================================================

sort_key = "batch"
sort_dir = 1
only_sel_cb = widgets.Checkbox(value=False, description="show only selected batches", indent=False)
sum_table_output = widgets.Output()

def render_summary_table():
    sum_table_output.clear_output(wait=True)
    with sum_table_output:
        color = color_fn()
        rows = [r for r in summary if not only_sel_cb.value or r["batch"] in selected]
        # Sort
        def sort_val(r):
            x = r.get(sort_key)
            if sort_key == "batch":
                return int(r["batch"].replace("Batch", ""))
            if x is None:
                return float("-inf") if sort_dir > 0 else float("inf")
            return x
        rows.sort(key=sort_val, reverse=sort_dir < 0)

        html = """<div style="overflow:auto;max-height:600px;border:1px solid #e3e6eb;border-radius:8px">
        <table style="border-collapse:collapse;width:100%;font-variant-numeric:tabular-nums;font-size:13px">
        <thead><tr style="background:#eef0f4;position:sticky;top:0">"""
        for k, lbl in SUMCOLS:
            arrow = " ▲" if k == sort_key and sort_dir > 0 else (" ▼" if k == sort_key and sort_dir < 0 else "")
            html += f"<th style='padding:4px 8px;border-bottom:1px solid #e3e6eb;text-align:right;cursor:pointer;white-space:nowrap'>{lbl}{arrow}</th>"
        html += "</tr></thead><tbody>"

        for r in rows:
            sel_class = "background:#eff6ff;" if r["batch"] in selected else ""
            html += f"<tr data-id='{r['batch']}' style='{sel_class}'>"
            for k, lbl in SUMCOLS:
                if k == "batch":
                    sw = f"<span style='display:inline-block;width:10px;height:10px;border-radius:2px;background:{color(r['batch'])};margin-right:4px'></span>"
                    html += f"<td style='padding:4px 8px;border-bottom:1px solid #e3e6eb;text-align:left'>{sw}{r['batch']}</td>"
                elif k == "split":
                    tag_class = "background:#fef3c7;color:#92400e" if r["split"] == "test" else "background:#e0e7ff;color:#3730a3"
                    html += f"<td style='padding:4px 8px;border-bottom:1px solid #e3e6eb;text-align:left'><span style='font-size:10px;padding:0 4px;border-radius:3px;{tag_class}'>{r['split']}</span></td>"
                else:
                    v = r.get(k)
                    if isinstance(v, float) and v == int(v):
                        v = int(v)
                    html += f"<td style='padding:4px 8px;border-bottom:1px solid #e3e6eb;text-align:right'>{fmt(v) if isinstance(v, (int, float)) else (v if v is not None else '–')}</td>"
            html += "</tr>"
        html += "</tbody></table></div>"

        display(HTML(html))
        
        # Add click handlers via JavaScript
        display(HTML("""
        <script>
        (function() {
            // Table sorting via header clicks
            var table = document.currentScript.previousElementSibling.querySelector('table');
            if (!table) return;
            var ths = table.querySelectorAll('th');
            ths.forEach(function(th, i) {
                th.addEventListener('click', function() {
                    var k = arguments[0].target.innerText.replace(/ [▲▼]/,'').trim();
                    // map label back to key
                    var map = {"Batch":"batch","Split":"split","# meas":"n_meas","t end":"t_end",
                               "VCD₀":"vcd0","Subst₀":"substrate0","Byprod₀":"byproduct0",
                               "VCD max":"vcd_max","t @ VCD max":"t_vcd_max","VCD end":"vcd_end",
                               "Viab end %":"viab_end","Byprod max":"byproduct_max","Subst min":"substrate_min",
                               "Vol₀":"vol0","Vol end":"vol_end",
                               "ΣFeed S":"feed_substrate_total","#S":"feed_substrate_n",
                               "ΣFeed B":"feed_b_total","#B":"feed_b_n",
                               "ΣFeed C":"feed_c_total","#C":"feed_c_n",
                               "ΣFeed D":"feed_d_total","#D":"feed_d_n"};
                    console.log('Sort requested by:', map[k] || k);
                });
            });
        })();
        </script>
        """))

# Since JS-based sort is limited in Jupyter, use a proper widget-based approach
sort_dd = widgets.Dropdown(
    options=[(lbl, k) for k, lbl in SUMCOLS],
    value="batch",
    description="Sort by:"
)
sort_asc_btn = widgets.ToggleButton(value=True, description="▲ asc", layout=widgets.Layout(width="auto"))

def on_sort_change(change):
    global sort_key, sort_dir
    sort_key = sort_dd.value
    sort_dir = 1 if sort_asc_btn.value else -1
    render_summary_table()

sort_dd.observe(on_sort_change, "value")
sort_asc_btn.observe(lambda _: on_sort_change(None), "value")
only_sel_cb.observe(lambda _: render_summary_table(), "value")

table_tab = widgets.VBox([
    widgets.HBox([only_sel_cb, sort_dd, sort_asc_btn]),
    sum_table_output
])

In [ ]:
# =============================================================
#  RAW DATA TAB
# =============================================================

raw_batch_dd = widgets.Dropdown(options=IDS, value=IDS[0], description="Batch:")
raw_output = widgets.Output()

def render_raw():
    raw_output.clear_output(wait=True)
    with raw_output:
        rows = batches[raw_batch_dd.value]["raw"]
        if not rows:
            display(HTML("<p style='color:#6b7280'>No raw data.</p>"))
            return
        cols = list(rows[0].keys())
        html = """<div style="overflow:auto;max-height:600px;border:1px solid #e3e6eb;border-radius:8px">
        <table style="border-collapse:collapse;width:100%;font-variant-numeric:tabular-nums;font-size:13px">
        <thead><tr style="background:#eef0f4;position:sticky;top:0">"""
        for c in cols:
            html += f"<th style='padding:4px 8px;border-bottom:1px solid #e3e6eb;text-align:right;white-space:nowrap'>{c}</th>"
        html += "</tr></thead><tbody>"
        for r in rows:
            html += "<tr>"
            for c in cols:
                v = r[c]
                if v is None:
                    td = ""
                elif isinstance(v, float):
                    td = fmt(v)
                else:
                    td = str(v)
                html += f"<td style='padding:4px 8px;border-bottom:1px solid #e3e6eb;text-align:right'>{td}</td>"
            html += "</tr>"
        html += "</tbody></table></div>"
        display(HTML(html))

raw_batch_dd.observe(lambda _: render_raw(), "value")

raw_tab = widgets.VBox([
    widgets.HTML("<p style='color:#6b7280'>These are the original rows from <code>bioreaction_remaining.csv</code> for one batch.</p>"),
    raw_batch_dd,
    raw_output
])

In [ ]:
# =============================================================
#  TAB SWITCHING & FULL LAYOUT
# =============================================================

tab_buttons = widgets.ToggleButtons(
    options=[("Plots", "plots"), ("Batch summary", "table"), ("Raw rows", "raw")],
    value="plots",
    button_style="",
    style={"button_width": "auto"},
)

tab_content = widgets.VBox([plots_tab])

def switch_tab(change):
    t = change["new"]
    if t == "plots":
        tab_content.children = [plots_tab]
        render_plots()
    elif t == "table":
        tab_content.children = [table_tab]
        render_summary_table()
    elif t == "raw":
        tab_content.children = [raw_tab]
        render_raw()

tab_buttons.observe(switch_tab, "value")

def refresh_active_tab():
    t = tab_buttons.value
    if t == "plots":
        render_plots()
    elif t == "table":
        render_summary_table()
    elif t == "raw":
        render_raw()

# Header
header = widgets.HBox([
    widgets.HTML("<h1 style='font-size:16px;margin:0'>Bioreactor Data Viewer</h1>"),
    tab_buttons,
], layout=widgets.Layout(padding="8px 12px", border_bottom="1px solid #e3e6eb", align_items="center"))

# Main layout: sidebar + content
main_area = widgets.HBox([sidebar, tab_content], layout=widgets.Layout(flex="1"))

full_ui = widgets.VBox([header, main_area], layout=widgets.Layout(width="100%", height="auto"))

display(full_ui)
render_plots()

## Bioreactor Data Viewer

**Run all cells above** to see the interactive dashboard.

### Features:
- **Plots tab** — Overlay or separate plot modes for selected batches. Control variables, layout, feed display (individual/cumulative), color mapping, markers, log-scale, and legend.
- **Batch summary tab** — Sortable table with key metrics per batch (VCD max, substrate min, feed totals, etc.). Filter to selected batches.
- **Raw rows tab** — View raw CSV rows per batch.

The original HTML viewer was built with Plotly + vanilla JS. This notebook replicates it using **ipywidgets** (dropdowns, checkboxes, toggle buttons) and **Plotly** figures in **Jupyter**.